
## Joins: Combine Retail Datasets:

The main business requirement is `to combine customers, orders, products, stores, and payments into a usable sales dataset`.

In [0]:
%run ./02_Data_Cleaning


## Joins are stared here:


### 1. Inner Join:

**Returns the rows are `matching in both DataFrames` based on the join condition.**

In [0]:
customer_orders_df = customers_clean_df.join(
    orders_clean_df,
    on="customer_id",
    how="inner"
)

display(customer_orders_df)


**Business question: Which customers have placed orders?**


## 2. Equi Join:

**An equi join `matches equal rows between columns`.**

In [0]:
equi_join_df = orders_clean_df.alias("o").join(
    customers_clean_df.alias("c"),
    F.col("o.customer_id") == F.col("c.customer_id"),
    "inner"
).select(
    "o.order_id",
    "o.customer_id",
    "c.customer_name",
    "o.order_status"
)

display(equi_join_df)


**Real-time use: Match orders to customer records using customer IDs.**


### 3. Left Outer Join: (Left Join)

- Returns `all rows from the left DataFrame` and `matching the rows from the right DataFrame`.
- If no match exists, the `right DataFrame columns contain NULL values`.

In [0]:
left_join_df = customers_clean_df.join(
    orders_clean_df,
    on="customer_id",
    how="left"
)

display(left_join_df)


**Business question: Which customers have orders, and which customers have never ordered?**


### 4. Right Outer Join: (Right Join)

- Returns `all rows from the right DataFram`e and `matching rows from the left DataFrame`.
- If no match exists, the `left DataFrame columns contain NULL values`.



In [0]:
right_join_df = customers_clean_df.join(
    orders_clean_df,
    on="customer_id",
    how="right"
)

display(right_join_df)


**Business use: Preserve all orders, including orders that have no matching customer record.**


### 5. Full Outer Join:

- Returns `all rows from both DataFrames`.
- Matching rows are `combined`, and non-matching rows contain `NULL values` for the missing side.

In [0]:
full_join_df = customers_clean_df.join(
    orders_clean_df,
    on="customer_id",
    how="full_outer"
)

display(full_join_df)


**Business use: Reconcile customer records against orders and identify unmatched records on either side.**


### 6. Left Semi Join:

- Returns the `rows from the left DataFrame` that have `at least one matching row` in the right DataFrame.
- Only `left DataFrame columns` are returned.

In [0]:
left_semi_df = customers_clean_df.join(
    orders_clean_df,
    on="customer_id",
    how="left_semi"
)

display(left_semi_df)


**Business question: Which customers have placed at least one order?**

### 7. Left Anti Join:

- Returns the `rows from the left DataFrame` that `do not have any matching row` in the right DataFrame.
- Only `left DataFrame columns` are returned.

In [0]:
left_anti_df = customers_clean_df.join(
    orders_clean_df,
    on="customer_id",
    how="left_anti"
)

display(left_anti_df)


**Business question: Which customers have never placed an order?**


### 8. Self Join:

Joins a `DataFrame with itself `using `aliases and a join condition`.

In [0]:
employees_data = [
    (1, "Ravi", None),
    (2, "Priya", 1),
    (3, "Arun", 1),
    (4, "Meena", 2)
]

employees_df = spark.createDataFrame(
    employees_data,
    ["employee_id", "employee_name", "manager_id"]
)

employee_hierarchy_df = employees_df.alias("e").join(
    employees_df.alias("m"),
    F.col("e.manager_id") == F.col("m.employee_id"),
    "left"
).select(
    F.col("e.employee_name").alias("employee"),
    F.col("m.employee_name").alias("manager")
)

display(employee_hierarchy_df)


**Business use: Identify which retail employees report to which managers.**


### 9. Cross Join / Cartesian Join:

Returns the `Cartesian product of two DataFrames`, combining `every row from the left DataFrame` with `every row from the right DataFrame`.

In [0]:
payment_methods_df = spark.createDataFrame(
    [("UPI",), ("Card",), ("Cash",)],
    ["payment_method"]
)

cross_join_df = stores_df.crossJoin(payment_methods_df)

display(cross_join_df)


**Business use: Generate all possible store-payment-method combinations for reporting or planning.**


### 10. Natural Join:

A natural join `automatically matches columns with the same name` and `returns one copy of the common join column`.

In [0]:
customers_clean_df.createOrReplaceTempView("customers")
orders_clean_df.createOrReplaceTempView("orders")

natural_join_df = spark.sql("""
    SELECT *
    FROM customers
    NATURAL INNER JOIN orders
""")

display(natural_join_df)


## Build the Retail Sales Dataset:

Now combine the datasets into a single detailed sales dataset.


### 1. Join Orders with Customers:

In [0]:
sales_step1_df = orders_clean_df.join(
    customers_clean_df.select(
        "customer_id",
        "customer_name",
        "city",
        "membership",
        "customer_segment"
    ),
    on="customer_id",
    how="inner"
)


### 2. Join with Order Items:

In [0]:
sales_step2_df = sales_step1_df.join(
    order_items_df,
    on="order_id",
    how="inner"
)


### 3. Join with Products:

In [0]:
sales_step3_df = sales_step2_df.join(
    products_clean_df,
    on="product_id",
    how="inner"
)


### 4. Join with Stores:

In [0]:
sales_step4_df = sales_step3_df.join(
    stores_df,
    on="store_id",
    how="left"
)


### 5. Join with Payments:

In [0]:
sales_df = sales_step4_df.join(
    payments_df,
    on="order_id",
    how="left"
)


### 6. Calculate Revenue:

- For this project, `calculate item revenue` using:

- **Net item revenue = (quantity × unit price) − discount**

In [0]:
sales_df = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).withColumn(
    "net_revenue",
    F.col("gross_revenue") - F.col("discount")
).withColumn(
    "order_month",
    F.date_format("order_date", "yyyy-MM")
)

display(sales_df)